In [9]:
# Setup
import torch
import copy
import json

from utils.data_reader import load_and_prepare_time_series_data
from utils.evaluator import Evaluator
from models.baseline_models import LSTMModel, BiLSTMModel, GRUModel
from models.custom_models import MGSSMModel, MGSSMsModel, ExtendedMGSSMsModel

In [3]:

selected_country_codes = ['US', 'IN', 'BR', 'FR', 'DE',
                         'GB', 'RU', 'IT', 'TR', 'ES',
                         'VN', 'AR', 'AU', 'AT', 'BD',
                         'BE', 'BG', 'CA', 'CL', 'CN',
                         'CU', 'DK', 'FI', 'GE', 'GR',
                         'ID', 'JP', 'JO', 'KE', 'KR',
                         'LR', 'MY','ML', 'MX', 'NL',
                         'NO', 'PH','SE', 'CH', 'TH']

# Fetch and prepare the data for each country code
with open("/home/theppawan/nn-models/data/COVID19_url_data.json", "r") as f:
    dataset = json.load(f)

train_loader_dist = {}
val_loader_dist = {}
test_loader_dist = {}
scaler_dist = {}
for key in selected_country_codes:
    train_loader, val_loader, test_loader, scaler = load_and_prepare_time_series_data(
        filepath_or_url = dataset['region'].format(region=key),
        target_column=['cumulative_confirmed'],
        date_column="date",
        seq_length=14,
        batch_size=64,
        train_split=0.8,
        fill_missing=True)
    train_loader_dist[key] = train_loader
    val_loader_dist[key] = val_loader
    test_loader_dist[key] = test_loader
    scaler_dist[key] = scaler

# Setup model configurations
model_config_dict = {
    "Baseline LSTM": LSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline BiLSTM": BiLSTMModel(input_size=1, hidden_size=256, num_layers=1, output_size=1),
    "Baseline GRU": GRUModel(input_size=1, hidden_size=128, num_layers=1, output_size=1),
    "MGSSM": MGSSMModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "MGSSMs": MGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32),
    "ExtendedMGSSMs": ExtendedMGSSMsModel(input_size=1, hidden_size=64, num_layers=1, output_size=1, gate_size=32, p=2)
}

Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/US.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/IN.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/BR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data/v3/location/FR.csv
Sorting data chronologicall by column: date
Succesfully loaded 991 sequential data points.
Prepared Training batches: 13 | Validation batches: 3
Fetching data from: https://storage.googleapis.com/covid19-open-data

In [4]:
trained_model_dict = {key: {} for key in selected_country_codes}
# loading the checkpointed models
for model_name, model_config in model_config_dict.items():
    # analyze the model from the checkpoint
    for key in selected_country_codes:
        country_specific_model = copy.deepcopy(model_config)
        country_specific_model.load_state_dict(torch.load(f"checkpoints/best_{model_config.__class__.__name__}_{key}.pth"))
        trained_model_dict[key][model_name] = country_specific_model

evaluator = Evaluator()
for key in selected_country_codes:
    val_loader = val_loader_dist[key]
    scaler = scaler_dist[key]
    evaluator.compare_models(trained_model_dict[key], val_loader, scaler)

Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating model: Baseline LSTM...
Evaluating model: Baseline BiLSTM...
Evaluating model: Baseline GRU...
Evaluating model: MGSSM...
Evaluating model: MGSSMs...
Evaluating model: ExtendedMGSSMs...
Evaluating mode

In [5]:
performance_data = evaluator.build_performance_dataframe(nested_models_dict=trained_model_dict, data_loaders_dict=val_loader_dist, scalers_dict=scaler_dist, target_metric="MSE")
performance_data

Evaluating models for dataset: US...
Evaluating models for dataset: IN...
Evaluating models for dataset: BR...
Evaluating models for dataset: FR...
Evaluating models for dataset: DE...
Evaluating models for dataset: GB...
Evaluating models for dataset: RU...
Evaluating models for dataset: IT...
Evaluating models for dataset: TR...
Evaluating models for dataset: ES...
Evaluating models for dataset: VN...
Evaluating models for dataset: AR...
Evaluating models for dataset: AU...
Evaluating models for dataset: AT...
Evaluating models for dataset: BD...
Evaluating models for dataset: BE...
Evaluating models for dataset: BG...
Evaluating models for dataset: CA...
Evaluating models for dataset: CL...
Evaluating models for dataset: CN...
Evaluating models for dataset: CU...
Evaluating models for dataset: DK...
Evaluating models for dataset: FI...
Evaluating models for dataset: GE...
Evaluating models for dataset: GR...
Evaluating models for dataset: ID...
Evaluating models for dataset: JP...
E

,Baseline LSTM,Baseline BiLSTM,Baseline GRU,MGSSM,MGSSMs,ExtendedMGSSMs
US,2.588301e+10,2.313354e+10,8.867755e+09,2.355023e+10,2.354199e+10,5.928214e+09
IN,1.293873e+07,2.914654e+07,3.726016e+07,1.100531e+08,1.813128e+08,2.886230e+08
BR,5.678985e+08,3.145638e+08,6.807692e+08,8.425402e+09,2.467321e+09,6.579640e+08
FR,9.472514e+12,4.914872e+12,8.067499e+12,3.519847e+12,4.291473e+12,3.915982e+12
DE,8.997218e+09,2.994709e+09,1.418149e+10,4.176153e+10,2.623893e+10,1.155571e+10
GB,2.379711e+08,3.753721e+08,1.191250e+09,8.128149e+08,6.317389e+08,1.702438e+08
RU,1.633875e+08,1.127623e+08,5.979336e+08,2.587188e+08,2.815312e+08,1.261225e+08
IT,9.266981e+08,7.145657e+08,9.292367e+09,7.346849e+08,1.682856e+09,6.576413e+08
TR,5.403155e+09,2.249364e+09,6.313348e+09,2.545855e+09,2.803937e+09,2.734768e+09
ES,2.753005e+08,1.730457e+08,1.199993e+09,1.246338e+09,3.624591e+08,2.688460e+08


In [6]:
friedman_statistic, p_value = evaluator.friedman_test(performance_data)
p_value

np.float64(0.0005822785735293359)

In [16]:
evaluator.holm_bonferroni_posthoc(performance_data, control_model="ExtendedMGSSMs", alpha=0.05, metric_is_loss=True)

,unadjusted p-value,Step (i),Holm threshold,Holm adjusted p-value,reject null hypothesis
baseline model,,,,,
MGSSMs,0.001319,1,0 0.010000 1 0.012500 2 0.016667 3 ...,0.006597,True
Baseline GRU,0.069340,2,0 0.010000 1 0.012500 2 0.016667 3 ...,0.277361,False
Baseline LSTM,0.276850,3,0 0.010000 1 0.012500 2 0.016667 3 ...,0.830549,False
MGSSM,0.304253,4,0 0.010000 1 0.012500 2 0.016667 3 ...,0.830549,False
Baseline BiLSTM,0.797531,5,0 0.010000 1 0.012500 2 0.016667 3 ...,0.830549,False
